# Speech - Sarawgi et al. 2020 on ADReSS
**Spec:** PLAN.md B1, FINDINGS 5, 7. **Runs on:** Colab, **no Drive**. ADReSS must not go to Kaggle (FINDINGS 7). Not run yet.

Nothing is mounted or stored on Drive. Colab's disk is wiped on every disconnect, so **each new session**
re-upload two files into Colab's left-hand *Files* panel (drag and drop into `/content`):

| File (made on your PC in `D:\Alzheimers\colab_upload\`) | Size |
|---|---|
| `speech_code.tar` (`common/`, `speech/`) | tiny |
| `speech_data.tar` (transcriptions + Full_wave_enhanced_audio) | ~660 MB |

Set the runtime to a GPU. Do NOT use Colab's Gemini / AI assistant with the data loaded.

In [ ]:
!pip install -q opensmile scikit-learn
# Run in EVERY new session: unpack what you uploaded into /content (see the table above)
import os, sys, subprocess
os.makedirs('/content/Alzheimers', exist_ok=True)
os.makedirs('/content/data/adress_train', exist_ok=True)
subprocess.run(['tar', '-xf', '/content/speech_code.tar', '-C', '/content/Alzheimers'], check=True)
subprocess.run(['tar', '-xf', '/content/speech_data.tar', '-C', '/content/data/adress_train'], check=True)
import numpy as np, pandas as pd, torch, pathlib
REPO = '/content/Alzheimers'     # folder containing common/ and speech/
DATA = '/content/data/adress_train'
sys.path.insert(0, REPO)
from speech.features import get_pause_features, get_intervention_features
from speech.models import MODELS
from speech.cv import cross_validate
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
from common.runlog import RunLog
log = RunLog('/content/out/speech.log')   # appends to a text file on Colab's temporary disk


## 1. Shape check (run first)

In [ ]:
with log.tee('shape check'):
    for name, shape in [('disfluency',(4,11)),('acoustic',(4,21)),('interventions',(4,32,3))]:
        m = MODELS[name](); m.train()
        out = m(torch.randn(*shape)); loss = out.sum() + m.penalty(); loss.backward()
        print(name, tuple(out.shape), 'ok')


> **SEND BACK TO CLAUDE:**
> the three lines printed (`disfluency ... ok`, etc.). If it errors: the last 10-15 lines of the error. STOP there.

## 2. Labels and features (108 subjects)

In [ ]:
with log.tee('labels and features'):
    root = pathlib.Path(DATA); rows = []
    for grp, label in [('cc', 0), ('cd', 1)]:
        for cha in sorted((root/'transcription'/grp).glob('*.cha')):
            rows.append(dict(id=cha.stem, label=label, cha=cha, wav=root/'Full_wave_enhanced_audio'/grp/f'{cha.stem}.wav'))
    df = pd.DataFrame(rows); print(df.label.value_counts().to_dict(), 'n =', len(df))   # expect 54/54
    disf = np.stack([get_pause_features(r.cha, r.wav) for r in df.itertuples()])
    inter = np.stack([get_intervention_features(r.cha) for r in df.itertuples()])
    print(disf.shape, inter.shape, 'any NaN/inf:', not np.isfinite(disf).all())


> **SEND BACK TO CLAUDE:**
> the one printed line with the label counts and n (expect 54/54, n = 108) and the shape line.

## 3. ComParE (6,373 functionals per file)
[OURS] `opensmile-python` `ComParE_2016` stands in for the official `SMILExtract -C IS13_ComParE.conf`
(same 6,373 feature count; exact equivalence not verified). Scaler + PCA(21) are fitted per fold inside `cross_validate`.

In [ ]:
with log.tee('ComParE'):
    import opensmile
    smile = opensmile.Smile(feature_set=opensmile.FeatureSet.ComParE_2016, feature_level=opensmile.FeatureLevel.Functionals)
    acoustic_raw = np.stack([smile.process_file(str(w)).values[0] for w in df.wav]); print(acoustic_raw.shape)  # (108, 6373)


> **SEND BACK TO CLAUDE:**
> the printed shape, expect (108, 6373). Slow cell - a few minutes is normal.

## 4. 5-fold CV (batch 8, Adam lr 0.01; inner-split checkpointing [OURS])

In [ ]:
with log.tee('5-fold CV'):
    feats = dict(disfluency=disf, acoustic_raw=acoustic_raw, interventions=inter)
    oof, fold_of = cross_validate(feats, df.label.values, seed=0, device=DEVICE)


> **SEND BACK TO CLAUDE:**
> nothing unless it errors (then the last 10-15 lines). It runs for a while.

## 5. Metrics — subject-level bootstrap, 95% CI, n and protocol

In [ ]:
with log.tee('metrics'):
    from common.metrics import balanced_acc_ci, f1_ci, auc_ci
    y = df.label.values; g = df.id.values; proto = '5-fold CV, n=108, stratified, seed 0'
    for k, v in oof.items():
        pred = (v >= 0.5).astype(int)
        print(k); print('  ', balanced_acc_ci(y, pred, proto, g)); print('  ', f1_ci(y, pred, proto, g))
        if k != 'ensemble_hard_vote': print('  ', auc_ci(y, v, proto, g))
    pd.DataFrame(dict(id=df.id, label=y, fold=fold_of, **oof)).to_csv('/content/out/speech_oof.csv', index=False)


> **SEND BACK TO CLAUDE:**
> EVERYTHING this cell prints (balanced accuracy / F1 / AUC lines for each network and the ensemble).

## 6. Export for the app (Disfluency + Acoustic)
Fits both networks on **all 108 subjects** and writes ONNX files. **Not an evaluation**: these 108 subjects are the training data; the only accuracy statement is the CV above. Interventions is research-only and is not exported. Needs `acoustic_raw` and `disf` from sections 2-3 in this same session.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'skl2onnx', 'onnxruntime', 'onnx'], check=True)
from speech.export import export
with log.tee('ONNX export'):
    export(disf, acoustic_raw, df.label.values, '/content/out', seed=0, device=DEVICE)


> **SEND BACK TO CLAUDE:** the three parity lines printed. Next cell downloads one zip: unzip `speech_results.zip`, save the `.onnx`/`.json` files to `D:\Alzheimers\models\` and the log/csv to `D:\Alzheimersesults\`.


## 7. Download the log and files


In [ ]:
# One zip = one download (browsers block several downloads at once)
import shutil
from google.colab import files
shutil.make_archive('/content/speech_results', 'zip', '/content/out')
print('Click Allow if the browser asks about downloads. If nothing downloads: Files panel > refresh > right-click speech_results.zip > Download.')
files.download('/content/speech_results.zip')
